In [25]:
!pip install selenium webdriver-manager beautifulsoup4 pandas tqdm openpyxl

In [26]:
import time
import re
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager

In [43]:
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

In [28]:
def crawl_hwahae_ranking(url, category_name):
    driver.get(url)
    time.sleep(3)

    for _ in range(18):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.7)

    text = driver.execute_script("return document.body.innerText;")
    lines = [x.strip() for x in text.split("\n") if x.strip()]

    start_idx = None
    for i, line in enumerate(lines):
        if "업데이트" in line:
            start_idx = i + 1
            break

    end_idx = None
    for i, line in enumerate(lines):
        if "사업자 정보" in line:
            end_idx = i
            break

    if start_idx is None or end_idx is None:
        print(category_name, "시작/끝 위치 못 찾음")
        return pd.DataFrame()

    rank_lines = lines[start_idx:end_idx]

    rows = []
    rank = 1
    i = 0

    while i < len(rank_lines) - 2 and rank <= 100:
        line = rank_lines[i]

        if line.isdigit():
            i += 1
            continue

        if (
            re.match(r"^[3-5]\.\d{2}$", rank_lines[i+1])
            and re.match(r"^[\d,]+$", rank_lines[i+2])
        ):
            rows.append({
                "category": category_name,
                "hwahae_rank": rank,
                "product_name": line,
                "hwahae_rating": float(rank_lines[i+1]),
                "hwahae_review": int(rank_lines[i+2].replace(",", ""))
            })

            rank += 1
            i += 3
        else:
            i += 1

    df_temp = pd.DataFrame(rows)
    print(category_name, "수집:", df_temp.shape)

    return df_temp

In [29]:
category_urls = {
    "skin_toner": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4156",
    "lotion_emulsion": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4165",
    "serum_ampoule": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4174",
    "cream": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4184",
    "mist": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4194",
    "skin_pad": "https://www.hwahae.co.kr/rankings?english_name=category&theme_id=4498"
}

In [30]:
hwahae_dfs = []

for category_name, url in category_urls.items():
    temp_df = crawl_hwahae_ranking(url, category_name)
    hwahae_dfs.append(temp_df)

hwahae_df = pd.concat(hwahae_dfs, ignore_index=True)

print("전체:", hwahae_df.shape)
hwahae_df.head()

skin_toner 수집: (100, 5)
lotion_emulsion 수집: (100, 5)
serum_ampoule 수집: (100, 5)
cream 수집: (100, 5)
mist 수집: (100, 5)
skin_pad 수집: (100, 5)
전체: (600, 5)


,category,hwahae_rank,product_name,hwahae_rating,hwahae_review
0,skin_toner,1,에스네이처아쿠아 오아시스 토너,4.74,27245
1,skin_toner,2,토리든다이브인 저분자 히알루론산 토너,4.70,34786
2,skin_toner,3,토니모리세라마이드 모찌 토너,4.69,17493
3,skin_toner,4,라운드랩1025 독도 토너,4.43,94669
4,skin_toner,5,이즈앤트리초저분자 히아루론산 토너,4.70,9436


In [31]:
hwahae_df.to_excel("hwahae_6category_top100.xlsx", index=False)
hwahae_df.to_csv("hwahae_6category_top100.csv", index=False, encoding="utf-8-sig")

In [32]:
hwahae_df["category"].value_counts()

category
skin_toner         100
lotion_emulsion    100
serum_ampoule      100
cream              100
mist               100
skin_pad           100
Name: count, dtype: int64

In [33]:
def parse_price_from_item(item):
    org_price_tag = item.select_one(".tx_org .tx_num")
    cur_price_tag = item.select_one(".tx_cur .tx_num")

    original_price = None
    sale_price = None

    if org_price_tag:
        original_price = int(org_price_tag.get_text(strip=True).replace(",", ""))

    if cur_price_tag:
        sale_price = int(cur_price_tag.get_text(strip=True).replace(",", ""))

    if sale_price is None:
        sale_price = original_price

    if original_price is None:
        original_price = sale_price

    discount_rate = None
    if original_price and sale_price:
        discount_rate = round((original_price - sale_price) / original_price * 100, 1)

    return original_price, sale_price, discount_rate

In [34]:
def search_oliveyoung_first_product(product_name):
    result = {
        "olive_search_query": product_name,
        "olive_found": 0,
        "olive_brand": None,
        "olive_product_name": None,
        "olive_full_name": None,
        "original_price": None,
        "sale_price": None,
        "discount_rate": None
    }

    try:
        driver.get("https://www.oliveyoung.co.kr/store/main/main.do")
        time.sleep(1.2)

        search_box = driver.find_element(By.ID, "query")
        search_box.clear()
        search_box.send_keys(product_name)
        search_box.send_keys(Keys.ENTER)
        time.sleep(2.3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        products = soup.select(".prd_info")

        if len(products) == 0:
            return result

        item = products[0]

        brand_tag = item.select_one(".tx_brand")
        name_tag = item.select_one(".tx_name")

        if brand_tag is None or name_tag is None:
            return result

        olive_brand = brand_tag.get_text(strip=True)
        olive_product_name = name_tag.get_text(" ", strip=True)

        original_price, sale_price, discount_rate = parse_price_from_item(item)

        result.update({
            "olive_found": 1,
            "olive_brand": olive_brand,
            "olive_product_name": olive_product_name,
            "olive_full_name": olive_brand + olive_product_name,
            "original_price": original_price,
            "sale_price": sale_price,
            "discount_rate": discount_rate
        })

        return result

    except Exception:
        return result

In [50]:
olive_results = []

In [51]:
from tqdm import tqdm

for idx, row in tqdm(hwahae_df.iterrows(), total=len(hwahae_df)):

    product_name = row["product_name"]

    result = search_oliveyoung_first_product(
        product_name
    )

    olive_results.append(result)

    # 20개마다 저장
    if (idx + 1) % 20 == 0:

        olive_df_temp = pd.DataFrame(olive_results)

        temp_final = pd.concat(
            [
                hwahae_df.iloc[:len(olive_df_temp)].reset_index(drop=True),
                olive_df_temp.reset_index(drop=True)
            ],
            axis=1
        )

        temp_final.to_excel(
            "temp_final_dataset.xlsx",
            index=False
        )

        print(
            f"{idx+1}개 저장 완료"
        )

    time.sleep(0.5)

  3%|▎         | 19/600 [01:51<1:00:10,  6.21s/it]

20개 저장 완료


  6%|▋         | 39/600 [03:53<56:05,  6.00s/it]  

40개 저장 완료


 10%|▉         | 59/600 [05:50<52:25,  5.81s/it]

...

 96%|█████████▋| 579/600 [56:07<01:59,  5.70s/it]

580개 저장 완료


100%|█████████▉| 599/600 [58:02<00:05,  5.76s/it]

600개 저장 완료


100%|██████████| 600/600 [58:07<00:00,  5.81s/it]


In [52]:
olive_df = pd.DataFrame(olive_results)

print(olive_df.shape)
olive_df.head()

(600, 8)


,olive_search_query,olive_found,olive_brand,olive_product_name,olive_full_name,original_price,sale_price,discount_rate
0,에스네이처아쿠아 오아시스 토너,1,에스네이처,[속촉촉 진정토너] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+젤크림 30ml),에스네이처[속촉촉 진정토너] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+젤...,25000.0,18900.0,24.4
1,토리든다이브인 저분자 히알루론산 토너,1,토리든,[NEW/6월올영픽] 토리든 다이브인 저분자 히알루론산 토너 500ml 기획 (+늘...,토리든[NEW/6월올영픽] 토리든 다이브인 저분자 히알루론산 토너 500ml 기획 ...,29000.0,19000.0,34.5
2,토니모리세라마이드 모찌 토너,0,None,None,None,NaN,NaN,NaN
3,라운드랩1025 독도 토너,1,라운드랩,[6월올영픽/미스트형] 라운드랩 1025 독도 토너 300ml 1+1 기획 (+늘어...,라운드랩[6월올영픽/미스트형] 라운드랩 1025 독도 토너 300ml 1+1 기획 ...,29800.0,25300.0,15.1
4,이즈앤트리초저분자 히아루론산 토너,1,이즈앤트리,이즈앤트리 초저분자 히아루론산 토너 300ml 리필기획 (+200ml 리필 + 미스...,이즈앤트리이즈앤트리 초저분자 히아루론산 토너 300ml 리필기획 (+200ml 리필...,33500.0,22500.0,32.8


In [53]:
final_df = pd.concat(
    [
        hwahae_df.reset_index(drop=True),
        olive_df.reset_index(drop=True)
    ],
    axis=1
)

print(final_df.shape)

(600, 13)


In [55]:
final_df.to_excel(
    "final_hwahae_oliveyoung_6category.xlsx",
    index=False
)

final_df.to_csv(
    "final_hwahae_oliveyoung_6category.csv",
    index=False,
    encoding="utf-8-sig"
)